In [1]:
LINK = 'https://www.youtube.com/watch?v=hv2ZIDrwmSw'

# PRIMO STEP ESTRAZIONE TRANSCRIPTION

In [1]:
from google import genai
from google.genai import types
import os
from google.genai.types import MediaResolution, ThinkingConfig, ThinkingLevel
from google.genai.types import (
    Content,
    CreateCachedContentConfig,
    FileData,
    GenerateContentConfig,
    Part,
)




In [2]:
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')
GEMINI_MODEL = 'gemini-3-flash-preview'
client = genai.Client(api_key=GEMINI_API_KEY)

In [65]:
PROMPT = "Trascrivi il seguente video youtube"

 N.B. Meglio mettere il file in cache: non ci sono riuscito mi da errore 400 invalid argument. Bisognerebbe penso scaricare il file e caricarlo su bucket come visto nell'esercitazione "gemini" del capitolo 5

In [70]:
video_file_data = FileData(
    file_uri=LINK,
    mime_type="video/mp4",
)
video = Part(file_data=video_file_data)
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[video, PROMPT],
)
with open('transcription.txt', 'w') as oFile:
    oFile.write(response.text)

In [3]:
with open('transcription.txt', 'r') as iFile:
    content = iFile.read()

In [4]:
content

'Il sistema hegeliano. Il sistema filosofico hegeliano probabilmente, ragazzi, è l\'ultimo grande sistema filosofico. Da Aristotele a Platone, Cartesio, Spinoza, i grandi filosofi del passato hanno elaborato dei sistemi filosofici onnicomprensivi. La grandezza della filosofia è stata anche quella di provare a dare una spiegazione totale della realtà. All\'interno del platonismo c\'è una risposta a tutto: dall\'astronomia alla fisica, dall\'etica alla politica, al linguaggio. Dentro l\'aristotelismo lo stesso, vi è tutto: vi è una teoria dei cieli, del movimento, vi è una teoria metafisica, vi è una teoria etica, vi è una teoria politica.\n\nHegel nell\'Ottocento è probabilmente l\'ultimo grande filosofo che ha l\'ambizione di elaborare un sistema filosofico in cui ogni cosa sia collocata al posto giusto. Mi piace definirlo ancora uno di quei filosofi dalla culla alla tomba, cioè se uno è hegeliano, se uno è platonico, se uno è aristotelico, se uno è spinoziano, ha una risposta, una let

# Step 2: costruiamo il grafo langraph: un agent che definisce il test, un agent che lo redige e un agent che fa reflection.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
gemini_model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key= GEMINI_API_KEY,
    temperature=0,
    top_k=1,
    max_tokens=None,
    timeout=None,
    max_retries=2.
)

In [9]:
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage
from typing import TypedDict, Annotated, Literal, List, Optional
from langgraph.graph import StateGraph, END
from langgraph.types import Command
from pydantic import BaseModel, Field

In [10]:

class ReviewAnalysis(BaseModel):
    """Analisi critica del test prodotto per decidere se approvare o iterare."""
    
    status: Literal["APPROVED", "REVISE_CONTENT", "REVISE_RESEARCH"] = Field(
        ..., 
        description="Stato attuale: APPROVED se pronto, REVISE_CONTENT se va riscritto, REVISE_RESEARCH se mancano dati."
    )
    
    score: int = Field(
        ..., ge=1, le=10, 
        description="Punteggio di qualità da 1 a 10."
    )
    
    critique: str = Field(
        ..., 
        description="Feedback dettagliato sugli errori o le mancanze riscontrate."
    )
    
    missing_info: Optional[List[str]] = Field(
        default=[], 
        description="Lista di argomenti o fatti specifici che il Researcher deve approfondire."
    )
    
    stylistic_notes: Optional[str] = Field(
        None, 
        description="Note su tono, linguaggio o struttura per il Writer."
    )

class AgentState(TypedDict):
    research_messages: Annotated[list, operator.add]
    lezione: str
    research_report: str
    review: Optional[ReviewAnalysis]
    revision_step: int = 0
    test: str


In [32]:
class TestMakerWorkflow:

    def __init__(self, system_message_researcher, system_message_test_writer, system_message_test_reflector, model, max_steps, tools = []):
        self.system_message_researcher = system_message_researcher
        self.system_message_test_writer = system_message_test_writer
        self.system_message_test_reflector = system_message_test_reflector
        self.base_model = model
        self.max_steps = max_steps
        self.researcher_model = model.bind_tools(tools)
        self.tools_dict = {t.name: t for t in tools}

        graph = StateGraph(AgentState) 
        graph.add_node('research', self.research)
        graph.add_node('write', self.write)
        graph.add_node('reflect', self.reflect)
        graph.add_node('research_actions', self.research_actions)
        graph.set_entry_point("research")
        self.graph = graph.compile()


    def research(self, state: AgentState):
        review = state.get('review', None)
        lezione = state.get('lezione', None)
        messages = state.get('research_messages', [])
        BASE_PROMPT = f"""Sei un Researcher esperto. Il tuo obiettivo è analizzare la trascrizione di una lezione 
        e arricchirne il contenuto con ricerche esterne (Tavily/Wikipedia).
        
        TRASCRIZIONE VIDEO:
        {lezione}
        
        ISTRUZIONI:
        1. Identifica i concetti chiave della lezione.
        2. Cerca dettagli tecnici, date, nomi o spiegazioni approfondite che non sono presenti 
           o sono poco chiari nella trascrizione.
        3. Fornisci un report dettagliato che il Writer userà per creare un test."""

        if review:
            missing_info = review.missing_info # E' una lista di stringhe
        if review:
            missing_info_str = "- " + "\n- ".join(review.missing_info)
            PROMPT_FOR_REVISION = f"""
            ATTENZIONE: Il Revisore ha scartato la bozza precedente.
            MOTIVAZIONE: {review.critique}
            
            DEVI APPROFONDIRE I SEGUENTI PUNTI MANCANTI:
            {missing_info_str}
            
            Esegui nuove ricerche specifiche per colmare queste lacune.
            """
        
        current_prompt = PROMPT_FOR_REVISION if review else BASE_PROMPT
        messages = [HumanMessage(content=current_prompt)] + messages
        if self.system_message_researcher:
            messages = [SystemMessage(content=self.system_message_researcher)] + messages        
        message = self.researcher_model.invoke(messages)
        

        if message.tool_calls:
            print(message.tool_calls)
            return Command(goto='research_actions', update = {'research_messages': [message]})
        else:
            content_text = message.content if isinstance(message.content, str) else message.content[0].get('text', '')        
            print(f"Researcher Output: {content_text}")
            return Command(goto='write', update = {'research_report': content_text})

    def research_actions(self, state:AgentState):
        tool_calls = state['research_messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            if not t['name'] in self.tools_dict:
                result = "bad tool name, retry"
            else:
                print(f"Sto chiamando {t['name']} con parametri {t['args']}")
                result = self.tools_dict[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        return Command(goto='research', update = {'research_messages': results})

    def write(self, state: AgentState):
        research_report = state['research_report']
        review = state.get('review', None)
        lezione = state.get('lezione', None)
        previous_test = state.get('test', "") 

        BASE_PROMPT = f"""
        Sei un esperto di instructional design. Il tuo compito è scrivere un test di valutazione.
        
        FONTE PRINCIPALE (Trascrizione video):
        {lezione}
        
        APPROFONDIMENTI DALLA RICERCA:
        {research_report}
        
        ISTRUZIONI:
        1. Crea 30 domande a scelta multipla basate sulla lezione.
        2. Usa gli 'APPROFONDIMENTI DALLA RICERCA' per rendere le domande tecnicamente precise.
        3. Fornisci 4 opzioni per domanda con una sola risposta corretta.
        4. Restituisci solo il testo del test.
        """

        # PROMPT 2: Revisione
        # Qui il Writer deve concentrarsi solo su ciò che il Revisore ha criticato
        if review:
            missing_info_str = "- " + "\n- ".join(review.missing_info) if review.missing_info else "Nessuna specifica."
            
            PROMPT_FOR_REVISION = f"""
            Stai revisionando un test che hai scritto in precedenza. 
            
            EGGEBOZZA PRECEDENTE DA CORRRE:
            {previous_test}
            
            CRITICA DEL REVISORE: 
            {review.critique}
            
            NOTE SULLO STILE E FORMATO:
            {review.stylistic_notes}
            
            INFORMAZIONI AGGIUNTIVE DA INTEGRARE:
            {missing_info_str}
            
            REPORT DI RICERCA AGGIORNATO (se necessario):
            {research_report}
            
            ISTRUZIONI:
            Non ricominciare da zero se non necessario. Mantieni le parti corrette della 'BOZZA PRECEDENTE' 
            e modifica o integra solo le parti criticate dal revisore.
            """
        current_prompt = PROMPT_FOR_REVISION if review else BASE_PROMPT
        
        messages = []
        if self.system_message_test_writer:
            messages.append(SystemMessage(content=self.system_message_test_writer))
        
        messages.append(HumanMessage(content=current_prompt))

        response = self.base_model.invoke(messages)
        
        test_out = response.content if isinstance(response.content, str) else response.content[0].get('text', '')
        print('TEST: '+test_out)
        return Command(goto='reflect', update={'test': test_out})

    def reflect(self, state: AgentState):
        lezione = state.get('lezione', "")
        test_da_revisionare = state.get('test', "")
        research_report = state.get('research_report', "")
        revision_step = state.get("revision_step", 0)
        if revision_step == self.max_steps:
            print("massimo numero di revisioni raggiunte")
            return Command(goto=END)
        REVIW_PROMPT = f"""
        Sei un Revisore Accademico. Il tuo compito è valutare il TEST generato basandoti sulla LEZIONE e sulla RICERCA.
        
        LEZIONE ORIGINALE:
        {lezione}
        
        REPORT DI RICERCA (Contesto extra):
        {research_report}
        
        TEST DA VALUTARE:
        {test_da_revisionare}
        
        CRITERI DI VALUTAZIONE:
        1. Accuratezza: Il test contiene allucinazioni o errori rispetto alla lezione?
        2. Completezza: Mancano concetti chiave fondamentali? (Se sì, usa REVISE_RESEARCH)
        3. Qualità Formale: Le domande sono chiare e le opzioni plausibili? (Se no, usa REVISE_CONTENT)
        4. Tono: il tono è accademico? (Se no, usa REVISE_CONTENT)
        
        Se il test è perfetto, imposta lo status su 'APPROVED'.
        """

        # Prepariamo i messaggi
        messages = []
        if self.system_message_test_reflector:
            messages.append(SystemMessage(content=self.system_message_test_reflector))
        
        messages.append(HumanMessage(content=REVIW_PROMPT))

        reflector_model = self.researcher_model.with_structured_output(ReviewAnalysis)
        result = reflector_model.invoke(messages)


        if result.status == 'APPROVED':
            print("Test Approvato!")
            return Command(goto=END, update={'review': result})
        
        elif result.status == 'REVISE_CONTENT':
            print(f"Revisione Contenuto: {result.critique}")
            return Command(goto='write', update={'review': result,"revision_step" : revision_step + 1})
        
        else:
            print(f"Nuova Ricerca Necessaria: {result.critique}")
            return Command(goto='research', update={'review': result, "revision_step" : revision_step + 1})        

In [12]:
from langchain.tools import tool
from tavily import TavilyClient

@tool
def tavily_search_tool(query: str, max_results: int = 5) -> str:
    """
    Esegue una ricerca web avanzata usando Tavily per trovare informazioni aggiornate,
    dati statistici o esempi reali non presenti nella trascrizione del video.
    
    Args:
        query (str): La query di ricerca ottimizzata.
        max_results (int): Numero di risultati (default 5).
        
    Returns:
        str: Un blocco di testo formattato contenente i risultati della ricerca.
    """
    # Recupera la chiave dalle variabili d'ambiente
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return "Errore: TAVILY_API_KEY non configurata."

    client = TavilyClient(api_key=api_key)
    
    try:
        # Eseguiamo la ricerca. 
        # 'search_depth="advanced"' è consigliato per compiti di ricerca complessi
        response = client.search(
            query=query, 
            max_results=max_results, 
            search_depth="advanced"
        )
        
        results = response.get('results', [])
        if not results:
            return f"Nessun risultato trovato sul web per: {query}"

        # Formattiamo l'output in una stringa leggibile per l'LLM
        formatted_results = [f"--- RISULTATI RICERCA WEB PER: {query} ---"]
        for res in results:
            content = f"TITOLO: {res.get('title')}\nURL: {res.get('url')}\nCONTENUTO: {res.get('content')}\n"
            formatted_results.append(content)
            
        return "\n".join(formatted_results)

    except Exception as e:
        return f"Errore durante la ricerca con Tavily: {str(e)}"

In [13]:
import wikipedia
from langchain_core.tools import tool

@tool
def search_wikipedia(query: str) -> str:
    """
    Cerca su Wikipedia per approfondire concetti tecnici, nomi propri, 
    date storiche o dettagli scientifici. 
    Input: una query di ricerca specifica.
    Output: un riassunto della pagina o un elenco di suggerimenti se la query è ambigua.
    """
    wikipedia.set_lang("it")
    
    try:
        page_content = wikipedia.summary(query, sentences=5)
        return f"Risultato Wikipedia per '{query}':\n\n{page_content}"
    
    except wikipedia.exceptions.DisambiguationError as e:
        return f"La ricerca per '{query}' è ambigua. Scegli uno dei seguenti argomenti: {', '.join(e.options[:5])}"
    
    except wikipedia.exceptions.PageError:
        return f"Nessuna pagina trovata su Wikipedia per '{query}'."
    
    except Exception as e:
        return f"Errore durante la ricerca su Wikipedia: {str(e)}"

In [14]:
system_message_researcher ="""Sei un Analista Ricercatore specializzato nel fact-checking e nell'approfondimento didattico. 
Il tuo compito è trasformare una trascrizione video spesso frammentaria in una base di conoscenza solida e tecnica.

REGOLE D'ORO:
1. Usa Wikipedia per definizioni enciclopediche, date e concetti teorici consolidati.
2. Usa Tavily per notizie recenti, dati statistici aggiornati o esempi pratici del mondo reale.
3. Non limitarti a ripetere il video: trova dettagli tecnici che l'oratore ha omesso ma che sono fondamentali per comprendere l'argomento.
4. Produci un "Research Report" strutturato, diviso per concetti chiave, pronto per essere usato da un Test Writer.
5. Se il Revisore ti segnala mancanze, focalizzati esclusivamente sui punti indicati nel campo 'missing_info'.
"""

system_message_test_writer = """
Sei un Instructional Designer esperto nella creazione di valutazioni per l'apprendimento. 
Il tuo obiettivo è creare test che non siano banali mnemonicamente, ma che testino la reale comprensione.

REGOLE D'ORO:
1. Basati sulla 'lezione' per il perimetro dei contenuti e sul 'research_report' per la precisione dei dettagli.
2. Struttura ogni domanda con: 1 risposta corretta e 3 "distrattori" (risposte sbagliate ma plausibili).
3. Evita risposte come "Tutte le precedenti" o "Nessuna delle precedenti".
4. Se ricevi una revisione, non cambiare l'intero test se non richiesto: correggi puntualmente le criticità segnalate mantenendo lo stile e la struttura della versione precedente.
"""
#5. Il tono deve essere formale e accademico.

system_message_critique = """
Sei un Supervisore della Qualità Didattica. Il tuo compito è garantire che il test prodotto sia impeccabile, accurato e utile. 
Agisci come un professore pignolo che deve dare l'approvazione finale.

CRITERI DI REVISIONE:
- CONTRADDIZIONI: Una domanda contraddice quanto detto nel video o nella ricerca? -> REVISE_CONTENT.
- LACUNE: Manca un intero capitolo della lezione nel test? -> REVISE_RESEARCH (specificando cosa cercare in 'missing_info').
- AMBIGUITÀ: Una domanda è formulata in modo che ci siano due risposte potenzialmente corrette? -> REVISE_CONTENT.
- PRECISIONE: Il test è troppo generico? (es. chiede "Di cosa parla il video?" invece di dettagli tecnici). -> REVISE_RESEARCH.

OUTPUT RICHIESTO:
Devi popolare sempre l'oggetto ReviewAnalysis. Sii estremamente specifico nel campo 'critique' affinché gli altri agenti sappiano esattamente come correggere il tiro.
"""

In [29]:
test_writer = TestMakerWorkflow(model = gemini_model, 
                                system_message_researcher=system_message_researcher, 
                                system_message_test_writer=system_message_test_writer, 
                                system_message_test_reflector=system_message_critique, 
                                max_steps = 3,
                                tools = [tavily_search_tool, search_wikipedia])

In [30]:
risultato = test_writer.graph.invoke({'lezione': content})

Researcher Output: ### **RESEARCH REPORT: IL SISTEMA FILOSOFICO DI G.W.F. HEGEL**
**Analista Ricercatore:** [AI Specialist]  
**Destinazione:** Test Writer (Preparazione materiale didattico/valutativo)

---

#### **1. Inquadramento Storico e Biografico (Integrazione Tecnica)**
Mentre la trascrizione cita Hegel come "egemonico" nella Germania dell'Ottocento, è necessario precisare le coordinate temporali e i luoghi chiave per contestualizzare il suo "titanismo".

*   **Dati Biografici:** Georg Wilhelm Friedrich Hegel (Stoccarda, 1770 – Berlino, 1831).
*   **Il Periodo d'Oro (Berlino):** Dal 1818 fino alla morte, Hegel occupò la cattedra che fu di Fichte all'Università di Berlino, diventando ufficialmente il "filosofo dello Stato prussiano".
*   **Il Conflitto con Schopenhauer:** L'episodio citato nel video avvenne nel **1820**. Arthur Schopenhauer, appena abilitato come libero docente, sfidò Hegel programmando le sue lezioni di "Filosofia intera" esattamente alla stessa ora di quelle di

In [31]:
risultato['revision_step']

1